# 中间件

## 1. 上下文管理 (代码生成)

In [1]:
import time
from pathlib import Path
from langchain.agents import create_agent

from deepagents.backends.filesystem import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemMiddleware
from deepagents.middleware.skills import SkillsMiddleware

In [2]:
SKILLS_DIR = Path("./skills")

GENERATED_DIR = Path("./generated")

GENERATED_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """
\
你是代码写手，一个专业的 Python 程序员智能体。

你的职责：接收用户的编程任务，生成完整且正确的 Python 代码，
并使用 write_file 工具将其保存为 Python 模块文件。

## 编码规范（必须严格遵守）

每个生成的 Python 模块必须包含：

1. **模块文档字符串** — 顶部一行概述模块用途
2. **类型标注** — 所有函数参数和返回值必须标注类型
3. **Google 风格文档字符串** — 每个公共函数必须有：
   ```
   def func(param: int) -> str:
       \"\"\"一行概述函数功能。

       Args:
           param: 参数描述。

       Returns:
           返回值描述。
       \"\"\"
   ```
4. **错误处理** — 验证输入参数，抛出具体异常
5. **入口点** — `if __name__ == "__main__":` 块包含冒烟测试
6. **snake_case 文件名**（如 `calculator.py`，而非 `Calculator.py`）

## 工作流程

1. **生成代码** — 编写包含用户所需全部函数的完整 Python 模块。
   不得编写不完整的代码或使用占位符。

2. **保存文件** — 调用 `write_file`：
   - `file_path`：必须以 `/generated/` 开头（如 `/generated/calculator.py`）
   - `content`：完整的 Python 源代码字符串

3. **报告** — 保存后告知用户：
   - 已保存的文件名
   - 代码功能的简要概述
   - 模块中定义的关键函数/类
   - 运行方式：`python <文件名>.py`

## 重要规则

- 必须将文件保存到 `/generated/` 路径下
- 不得编写不完整的代码，不得在函数体中使用 `pass` 或 `...` 占位符
- 不得在文件内容中包含 markdown 代码围栏（```）
- 必须生成完整的、可直接运行的 Python 代码
- 如用户要求多个函数，将它们全部放在一个模块文件中
- 生成代码后立即调用 write_file 保存——不要先解释
"""

In [10]:
fs_backend = FilesystemBackend(
    root_dir="./backend",
    virtual_mode=True,
    max_file_size_mb=10
)

fs_middleware = FilesystemMiddleware(
    backend=fs_backend,
    tools=["ls", "read_file", "write_file", "edit_file"]
)

fs_skill = SkillsMiddleware(
    backend=fs_backend,
    sources=["/skills/"]
)

In [11]:
agent = create_agent(
    model="ollama:qwen3.5:0.8b",
    tools=[],
    middleware=[fs_middleware, fs_skill],
    system_prompt=SYSTEM_PROMPT
)

In [12]:
outputs = agent.invoke(
    input={
        "messages": [
            {"role": "user", "content": "写一个冒泡算法，保存为maopao.py。"}
        ]
    },
    config={
        "recursion_limit": 50
    }
)

In [ ]:
outputs

## 2. 计划与分发

In [16]:
from deepagents.backends import StateBackend
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.tools import tool

### 2.1 计划

In [18]:
backend = StateBackend()

agent = create_agent(
    model="ollama:qwen2.5:7b-instruct-q4_K_M",
    tools=[],
    middleware=[TodoListMiddleware()],
)

user_prompt = "开发一个智能聊天程序"

result = agent.invoke(
    input={
        "messages": [
            {"role": "user", "content": user_prompt}
        ]
    }
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

开发一个智能聊天程序
================================== Ai Message ==================================

为了开发一个智能聊天程序，我们可以将其分解为以下几个步骤：

1. **需求分析**：确定聊天程序的功能需求，例如支持的语言、对话类型等。
2. **架构设计**：设计系统的总体架构，包括前端、后端和通信协议。
3. **选择技术栈**：选择适合开发的编程语言和框架。
4. **环境搭建**：搭建开发环境，包括安装必要的软件和依赖库。
5. **编写代码**：开始编写核心功能的代码。
6. **集成测试**：进行单元测试和集成测试，确保各个模块正常工作。
7. **部署上线**：将程序部署到服务器，确保程序可以正常运行。
8. **维护与优化**：根据用户反馈进行维护和优化。

现在，我将这些步骤添加到待办事项列表中，并标记第一个步骤为正在进行中。
Tool Calls:
  write_todos (82ed0bb0-83a3-4fee-b1db-74e34fc9fb60)
 Call ID: 82ed0bb0-83a3-4fee-b1db-74e34fc9fb60
  Args:
    todos: [{'content': '需求分析：确定聊天程序的功能需求，例如支持的语言、对话类型等。', 'status': 'in_progress'}, {'content': '架构设计：设计系统的总体架构，包括前端、后端和通信协议。', 'status': 'pending'}, {'content': '选择技术栈：选择适合开发的编程语言和框架。', 'status': 'pending'}, {'content': '环境搭建：搭建开发环境，包括安装必要的软件和依赖库。', 'status': 'pending'}, {'content': '编写代码：开始编写核心功能的代码。', 'status': 'pending'}, {'content': '集成测试：进行单元测试和集成测试，确保各个模块正常工作。', 'st

### 2.2 分发

In [ ]:
from deepagents.backends import StateBackend
from deepagents.middleware import SubAgentMiddleware
from langchain.agents import create_agent
from langchain.tools import tool